# Text classification with representation models

In [1]:
from datasets import load_dataset

# load data
data = load_dataset("rotten_tomatoes")
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [2]:
data["train"][0, 1]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-earth .'],
 'label': [1, 1]}

## Using a task-specific model

In [36]:
from transformers import pipeline

# Path to our HF model
# model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"
model_path = "distilbert-base-uncased-finetuned-sst-2-english"

# Load model into pipeline
pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    device="cuda:0"
)

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [37]:
print(pipe.model.num_parameters() / 10**6)

66.95501


In [12]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

# Run inference
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    # print(output)
    # break
    negative_score = output[0]["score"]
    positive_score = output[1]["score"]
    # positive_score = output[2]["score"] # model also has a neutral label
    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

100%|██████████| 1066/1066 [00:06<00:00, 169.83it/s]


In [13]:
from sklearn.metrics import classification_report

def evaluate_performance(y_true, y_pred):
    """Create and print the classification report"""
    performance = classification_report(
        y_true, y_pred,
        target_names=["Negative Review", "Positive Review"]
    )
    print(performance)

In [14]:
evaluate_performance(data["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.89      0.90      0.90       533
Positive Review       0.90      0.89      0.90       533

       accuracy                           0.90      1066
      macro avg       0.90      0.90      0.90      1066
   weighted avg       0.90      0.90      0.90      1066



## Using an embedding model

In [15]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to embeddings
train_embeddings = model.encode(data["train"]["text"], show_progress_bar=True)
test_embeddings = model.encode(data["test"]["text"], show_progress_bar=True)

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Batches:   0%|          | 0/267 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

In [17]:
print(train_embeddings.shape)
print(test_embeddings.shape)

(8530, 768)
(1066, 768)


In [18]:
from sklearn.linear_model import LogisticRegression

# Train a logistic regression on our train embeddings
clf = LogisticRegression(random_state=42)
clf.fit(train_embeddings, data["train"]["label"])

LogisticRegression(random_state=42)

In [19]:
# Predict previously unseen instances
y_pred = clf.predict(test_embeddings)
evaluate_performance(data["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.85      0.86      0.85       533
Positive Review       0.86      0.85      0.85       533

       accuracy                           0.85      1066
      macro avg       0.85      0.85      0.85      1066
   weighted avg       0.85      0.85      0.85      1066



### What if we do not have labeled data

In [24]:
# Create embeddings for our labels
label_embeddings = model.encode(["A negative movie review", "A positive movie review"])

In [25]:
# Compute cosine similarity and assign best label to docs

from sklearn.metrics.pairwise import cosine_similarity

# Find the best matching label for each document
sim_matrix = cosine_similarity(test_embeddings, label_embeddings)
y_pred = np.argmax(sim_matrix, axis=1)

evaluate_performance(data["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.83      0.76      0.79       533
Positive Review       0.78      0.85      0.81       533

       accuracy                           0.80      1066
      macro avg       0.80      0.80      0.80      1066
   weighted avg       0.80      0.80      0.80      1066



# Text classification with Generative models

In [26]:
# laod our model
pipe = pipeline(
    "text2text-generation", 
    model="google/flan-t5-small", 
    device="cuda:0"
)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [31]:
# Prepare our data
# We cannot simply get classificatio nresults from gen models

prompt = "Is the following movie review positive or nagative?"
data = data.map(lambda example: {"t5": prompt + example['text']})
data
# new_data = [{"t5": prompt + example["text"]} for example in data]
# new_data

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label', 't5'],
        num_rows: 1066
    })
})

In [32]:
# Run inference
y_pred = []
for output in tqdm(pipe(KeyDataset(data["test"], "t5")), total=len(data["test"])):
    text = output[0]["generated_text"]
    y_pred.append(0 if text == "negative" else 1)
    
evaluate_performance(data["test"]["label"], y_pred)

  0%|          | 0/1066 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
100%|██████████| 1066/1066 [00:35<00:00, 30.20it/s]

                 precision    recall  f1-score   support

Negative Review       0.86      0.76      0.81       533
Positive Review       0.78      0.88      0.83       533

       accuracy                           0.82      1066
      macro avg       0.82      0.82      0.82      1066
   weighted avg       0.82      0.82      0.82      1066



In [35]:
print(pipe.model.num_parameters() / 10**6)

76.961152


We see that the inference is slow. This is because our model size is larger and also input/output length is greater.

## ChatGPT for classification

In [65]:
import os
# import openai
import cohere
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)

# Create client
client = cohere.Client(os.getenv("COHERE_API_KEY"))

In [66]:
def coheregpt_generation(prompt, document, model="gpt-3.5-turbo-0125"):
    """Generate an output based on a prompt and an input document."""
    messages=[
        {
            "role": "system",
            "message": "You are a helpful assistant."
            },
        {
            "role": "user",
            "message":   prompt.replace("[DOCUMENT]", document)
            }
    ]

    chat_completion = client.chat(
	message= f"{prompt.replace('[DOCUMENT]', document)}"
    )
    return chat_completion

In [67]:
# Define a prompt template as a base
prompt = """
Predict whether the following document is a positive or negative movie review:

[DOCUMENT]

If it is positive return 1 and if it is negative return 0. Do not give any other answers.
"""

In [73]:
# Predict the target using CohereGPT
document = "It is splendid."
# document = "It is horrible."
print(coheregpt_generation(prompt, document))

text='1' generation_id='92908b90-f2a0-4066-97ad-d85f490ea65c' citations=None documents=None is_search_required=None search_queries=None search_results=None finish_reason='COMPLETE' tool_calls=None chat_history=[Message_User(message='\nPredict whether the following document is a positive or negative movie review:\n\nIt is splendid.\n\nIf it is positive return 1 and if it is negative return 0. Do not give any other answers.\n', tool_calls=None, role='USER'), Message_Chatbot(message='1', tool_calls=None, role='CHATBOT')] prompt=None meta=ApiMeta(api_version=ApiMetaApiVersion(version='1', is_deprecated=None, is_experimental=None), billed_units=ApiMetaBilledUnits(input_tokens=46, output_tokens=1, search_units=None, classifications=None), tokens=ApiMetaTokens(input_tokens=110, output_tokens=1), warnings=None) response_id='5ca5c7b5-3dbd-4320-8d2b-35cb951f9407'


In [69]:
from datasets import load_dataset

# load data
data = load_dataset("rotten_tomatoes")

In [70]:
# Using only 5 example due to API call limit
data = data["test"][:5]
predictions = [coheregpt_generation(prompt, doc) for doc in tqdm(data["text"])]
results = {}

for i, (doc, prediction) in enumerate(zip(data["text"], predictions)):
    results[i] = {
        "doc": doc,
        "prediction": prediction.text
    }

print(results)

# # Extract predictions
# y_pred = [int(pred) for pred in predictions]

# # Evaluate performance
# evaluate_performance(data["test"]["label"], y_pred)

100%|██████████| 5/5 [00:01<00:00,  3.88it/s]

{0: {'doc': 'lovingly photographed in the manner of a golden book sprung to life , stuart little 2 manages sweetness largely without stickiness .', 'prediction': '1'}, 1: {'doc': 'consistently clever and suspenseful .', 'prediction': '1'}, 2: {'doc': 'it\'s like a " big chill " reunion of the baader-meinhof gang , only these guys are more harmless pranksters than political activists .', 'prediction': '1'}, 3: {'doc': 'the story gives ample opportunity for large-scale action and suspense , which director shekhar kapur supplies with tremendous skill .', 'prediction': '1'}, 4: {'doc': 'red dragon " never cuts corners .', 'prediction': '1'}}
